# Machine Learning End-to-End Project — Loan Default Dataset

This notebook builds a complete **Machine Learning Classification Model step by step using a real CSV dataset**.

### Business Problem
Predict whether a customer will default on a loan.

### Target
`loan_default`

- `0` → No default
- `1` → Default

### Complete ML Flow

**Problem Definition → Data Collection → Data Understanding → EDA → Data Cleaning → Outlier Detection → Feature Engineering → Feature Selection → Train/Test Split → Preprocessing → Model Selection → Model Training → Prediction → Evaluation → Hyperparameter Tuning → Final Model → Save Model**


## 1. Import Libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

import warnings
warnings.filterwarnings("ignore")


## 2. Load the Dataset

Place `loan_default_dataset.csv` in the same folder as this notebook.


In [2]:
df = pd.read_csv("loan_default_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())


Dataset loaded successfully!
Shape: (148670, 34)


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


## 3. Understand the Dataset


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()


In [3]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,148670.0,NaN,NaN,NaN,99224.5,42917.476598,24890.0,62057.25,99224.5,136391.75,173559.0
year,148670.0,NaN,NaN,NaN,2019.0,0.0,2019.0,2019.0,2019.0,2019.0,2019.0
loan_limit,145326,2,cf,135348,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,148670,4,Male,42346,NaN,NaN,NaN,NaN,NaN,NaN,NaN
approv_in_adv,147762,2,nopre,124621,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_type,148670,3,type1,113173,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_purpose,148536,4,p3,55934,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Credit_Worthiness,148670,2,l1,142344,NaN,NaN,NaN,NaN,NaN,NaN,NaN
open_credit,148670,2,nopc,148114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
business_or_commercial,148670,2,nob/c,127908,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Exploratory Data Analysis (EDA)

We will perform:

- Univariate analysis
- Target analysis
- Categorical analysis
- Correlation analysis
- Distribution analysis


In [4]:
# Missing values
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))

# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Target distribution
print("\nTarget distribution:")
display(df["loan_default"].value_counts())

print("\nTarget percentage:")
display(df["loan_default"].value_counts(normalize=True).mul(100).round(2))


Missing values:


Upfront_charges              39642
Interest_rate_spread         36639
rate_of_interest             36439
dtir1                        24121
LTV                          15098
property_value               15098
income                        9150
loan_limit                    3344
approv_in_adv                  908
submission_of_application      200
age                            200
loan_purpose                   134
Neg_ammortization              121
term                            41
Region                           0
total_units                      0
Security_Type                    0
Status                           0
co-applicant_credit_type         0
Credit_Score                     0
credit_type                      0
ID                               0
Secured_by                       0
occupancy_type                   0
construction_type                0
year                             0
interest_only                    0
loan_amount                      0
business_or_commerci

Duplicate rows: 0

Target distribution:


KeyError: 'loan_default'

In [ ]:
# Target distribution
sns.countplot(data=df, x="loan_default")
plt.title("Loan Default Distribution")
plt.show()


In [ ]:
# Numeric distributions
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()


In [ ]:
# Categorical variables
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="education", ax=axes[0])
axes[0].set_title("Education Distribution")
axes[0].tick_params(axis="x", rotation=20)

sns.countplot(data=df, x="employment_type", ax=axes[1])
axes[1].set_title("Employment Type Distribution")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
# Default rate by categorical features
print("Default rate by education:")
display(df.groupby("education")["loan_default"].mean().sort_values(ascending=False))

print("Default rate by employment type:")
display(df.groupby("employment_type")["loan_default"].mean().sort_values(ascending=False))


In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 7))
sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Matrix")
plt.show()


## 5. Data Cleaning

We have intentionally included a small number of missing values and duplicate rows so that the cleaning stage can be practiced.

### Cleaning steps
1. Remove duplicate rows
2. Handle missing numeric values with median
3. Handle missing categorical values with most frequent value
4. Keep the target column separate


In [ ]:
print("Before cleaning:", df.shape)

df = df.drop_duplicates().copy()

print("After removing duplicates:", df.shape)
print("\nRemaining missing values:")
display(df.isnull().sum())


## 6. Outlier Detection

Use the IQR method to inspect numeric outliers.

We will **not blindly delete all outliers**, because some extreme financial values may be genuine observations.


In [ ]:
numeric_features = [
    "age",
    "income",
    "credit_score",
    "loan_amount",
    "employment_years",
    "debt_ratio",
    "account_balance",
    "existing_loans"
]

Q1 = df[numeric_features].quantile(0.25)
Q3 = df[numeric_features].quantile(0.75)
IQR = Q3 - Q1

outlier_counts = (
    (df[numeric_features] < Q1 - 1.5 * IQR) |
    (df[numeric_features] > Q3 + 1.5 * IQR)
).sum()

display(outlier_counts.sort_values(ascending=False))


## 7. Feature Engineering

Create a meaningful financial feature:

`loan_to_income_ratio = loan_amount / income`

A higher ratio can indicate greater borrowing relative to income.


In [ ]:
df["loan_to_income_ratio"] = df["loan_amount"] / (df["income"] + 1)

display(
    df[["loan_amount", "income", "loan_to_income_ratio"]].head()
)


## 8. Feature-Target Separation

`X` contains independent variables.

`y` contains the target variable.


In [ ]:
X = df.drop(columns="loan_default")
y = df["loan_default"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 9. Train-Test Split

We split the dataset **before fitting preprocessing steps** to prevent data leakage.

`stratify=y` keeps approximately the same class distribution in train and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))


## 10. Identify Numeric and Categorical Features


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


## 11. Preprocessing Pipeline

### Numeric pipeline
- Fill missing values using median
- Standardize using `StandardScaler`

### Categorical pipeline
- Fill missing values using most frequent value
- Apply One-Hot Encoding

Using a pipeline ensures preprocessing is learned from training data only.


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessing pipeline created.")


## 12. Model Selection

We compare three common classification algorithms:

1. Logistic Regression
2. Decision Tree
3. Random Forest


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}


## 13. Build Training Pipelines

Each model receives the **same preprocessing pipeline**, followed by the model.


In [ ]:
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

print("Pipelines created:")
for name in pipelines:
    print("-", name)


## 14. Train the Models

This is where the machine learning algorithms learn patterns from the training data.


In [ ]:
trained_models = {}

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    print(f"{name} trained successfully.")


## 15. Make Predictions


In [ ]:
predictions = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test)
    print(f"\n{name}")
    print("First 10 predictions:", predictions[name][:10])


## 16. Model Evaluation

We compare:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC


In [ ]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values(
    by="F1",
    ascending=False
)

display(results_df)


## 17. Classification Report


In [ ]:
for name, model in trained_models.items():
    print("=" * 70)
    print(name)

    y_pred = model.predict(X_test)

    print(classification_report(y_test, y_pred))


## 18. Confusion Matrix

The confusion matrix shows:

- True Negative
- False Positive
- False Negative
- True Positive


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]

best_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_name}")

plt.show()


## 19. ROC Curve


In [ ]:
plt.figure(figsize=(8, 6))

for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC={auc:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()


## 20. Hyperparameter Tuning

Now tune the Random Forest using `GridSearchCV`.

Cross-validation helps estimate performance across multiple training/validation folds.


In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("\nBest cross-validation F1:")
print(grid.best_score_)


## 21. Evaluate the Tuned Model on Unseen Test Data


In [ ]:
tuned_model = grid.best_estimator_

y_pred_tuned = tuned_model.predict(X_test)
y_prob_tuned = tuned_model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall   :", recall_score(y_test, y_pred_tuned))
print("F1       :", f1_score(y_test, y_pred_tuned))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_tuned))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))


## 22. Feature Importance

Because preprocessing includes one-hot encoding, the transformed feature names must be retrieved before interpreting Random Forest feature importance.


In [ ]:
rf_model = tuned_model.named_steps["model"]
fitted_preprocessor = tuned_model.named_steps["preprocessor"]

feature_names = fitted_preprocessor.get_feature_names_out()

importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

display(importance.head(15).to_frame("importance"))


In [ ]:
importance.head(15).sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.title("Top Feature Importances")
plt.xlabel("Importance")
plt.show()


## 23. Final Model

The tuned Random Forest becomes the final model for this example.

In a real project, final model selection should also consider business cost, interpretability, latency, fairness, calibration and operational requirements.


In [ ]:
final_model = tuned_model

print(final_model)


## 24. Predict a New Customer

The new record must contain the same input columns used by the model.


In [ ]:
new_customer = pd.DataFrame([{
    "age": 35,
    "income": 75000,
    "credit_score": 720,
    "loan_amount": 180000,
    "employment_years": 8,
    "debt_ratio": 0.30,
    "account_balance": 50000,
    "existing_loans": 1,
    "education": "Graduate",
    "employment_type": "Salaried",
    "loan_to_income_ratio": 180000 / 75000
}])

prediction = final_model.predict(new_customer)[0]
probability = final_model.predict_proba(new_customer)[0, 1]

print("Prediction:", prediction)
print("Default probability:", round(probability, 4))

if prediction == 1:
    print("Result: Customer is predicted to DEFAULT.")
else:
    print("Result: Customer is predicted to NOT DEFAULT.")


## 25. Save the Final Model


In [ ]:
model_path = "loan_default_final_model.pkl"

joblib.dump(final_model, model_path)

print("Model saved successfully:", model_path)


## 26. Load the Saved Model and Predict


In [ ]:
loaded_model = joblib.load("loan_default_final_model.pkl")

loaded_prediction = loaded_model.predict(new_customer)[0]
loaded_probability = loaded_model.predict_proba(new_customer)[0, 1]

print("Loaded model prediction:", loaded_prediction)
print("Loaded model probability:", round(loaded_probability, 4))


# Complete ML Workflow — Interview Revision

```text
1. Problem Definition
        ↓
2. Data Collection
        ↓
3. Data Understanding
        ↓
4. EDA
        ↓
5. Data Cleaning
        ↓
6. Outlier Detection
        ↓
7. Feature Engineering
        ↓
8. Feature Selection
        ↓
9. X / y Separation
        ↓
10. Train-Test Split
        ↓
11. Missing Value Imputation
        ↓
12. Encoding
        ↓
13. Feature Scaling
        ↓
14. Model Selection
        ↓
15. Model Training
        ↓
16. Prediction
        ↓
17. Evaluation
        ↓
18. Cross-Validation
        ↓
19. Hyperparameter Tuning
        ↓
20. Final Model Selection
        ↓
21. Save Model
        ↓
22. Deployment
        ↓
23. Monitoring
```

### Key Interview Point

**Data leakage must be avoided.** Split the data first, then fit preprocessing on training data only. Pipelines and `ColumnTransformer` make this workflow safer and reproducible.
